# Lab · Logistic regression: dự đoán · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 6, Bài 1). Theo lab "Logistic Regression" của khóa: hàm sigmoid, dựng mô hình bằng `nn.Sequential`, bằng lớp `nn.Module`, và ghép nhiều khối.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. Hàm sigmoid

`z = torch.arange(-100, 100, 0.1).view(-1, 1)`.
- `s1`: sigmoid của `z` bằng đối tượng `nn.Sigmoid()`;
- `s2`: sigmoid của `z` bằng hàm `torch.sigmoid`;
- `same`: hai kết quả có trùng nhau không;
- `s_at_0`: sigmoid tại `z = 0` (số Python).

Vẽ `s1` theo `z`: bạn sẽ thấy đường chữ S.

In [ ]:
z = torch.arange(-100, 100, 0.1).view(-1, 1)
s1 = nn.Sigmoid()(z)
s2 = torch.sigmoid(z)
same = torch.allclose(s1, s2)
s_at_0 = torch.sigmoid(torch.tensor(0.0)).item()
plt.plot(z.numpy(), s1.numpy()); plt.xlabel("z"); plt.ylabel("sigmoid(z)"); plt.show()
print(same, s_at_0)

In [ ]:
assert same is True
assert s_at_0 == 0.5
assert s1[0].item() < 1e-6 and s1[-1].item() > 1 - 1e-6, "z rất âm → 0, z rất dương → 1"
assert 0.0 <= s1.min().item() and s1.max().item() <= 1.0
print("✅ Bài 1 đúng")

## Bài 2. Từ xác suất đến nhãn bằng ngưỡng

`z = [-3., -1., 2., 5.]`.
- `p`: xác suất (sigmoid của `z`);
- `labels`: nhãn dự đoán dùng ngưỡng: `p ≥ 0.5` → 1, ngược lại 0 (kiểu `float`);
- `labels_z`: nhãn lấy trực tiếp từ dấu của `z` (`z > 0` → 1);
- `agree`: hai cách cho cùng nhãn không?

Câu hỏi: vì sao hai cách luôn trùng nhau (trừ trường hợp đúng bằng ngưỡng)?

In [ ]:
z = torch.tensor([-3.0, -1.0, 2.0, 5.0])
p = torch.sigmoid(z)
labels = (p >= 0.5).float()
labels_z = (z > 0).float()
agree = torch.equal(labels, labels_z)
print(p, labels, labels_z, agree)

In [ ]:
assert labels.tolist() == [0.0, 0.0, 1.0, 1.0]
assert agree is True
print("✅ Bài 2 đúng: ngưỡng 0.5 trên p chính là ranh giới z = 0")

## Bài 3. `nn.Sequential` với một đầu vào

Đặt `torch.manual_seed(2)`. Tạo `model = nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())`.
- `keys`: danh sách key của `model.state_dict()`;
- `yhat1`: dự đoán cho `x = [[1.0]]`;
- `yhat2`: dự đoán cho `X = [[1.0], [100.0]]`.
Đoán trước: xác suất của mẫu `100.0` sẽ gần giá trị nào, và vì sao?

In [ ]:
torch.manual_seed(2)
model = nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())
keys = list(model.state_dict().keys())
yhat1 = model(torch.tensor([[1.0]]))
yhat2 = model(torch.tensor([[1.0], [100.0]]))
print(keys, yhat1, yhat2)

In [ ]:
assert keys == ["0.weight", "0.bias"]
assert tuple(yhat1.shape) == (1, 1) and tuple(yhat2.shape) == (2, 1)
assert 0 < yhat1.item() < 1
assert yhat2[1].item() > 0.99, "đầu vào 100 làm sigmoid bão hòa về 1 (với tham số seed 2)"
print("✅ Bài 3 đúng")

## Bài 4. `nn.Sequential` với hai đầu vào

Tạo `model2 = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())`.
- `p_single`: dự đoán cho `x = [[1.0, 1.0]]`;
- `p_batch`: dự đoán cho `X = [[1., 1.], [1., 2.], [1., 3.]]`.
Lưu `shapes = (p_single.shape, p_batch.shape)`.

In [ ]:
model2 = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
p_single = model2(torch.tensor([[1.0, 1.0]]))
p_batch = model2(torch.tensor([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]))
shapes = (p_single.shape, p_batch.shape)
print(shapes)

In [ ]:
assert tuple(shapes[0]) == (1, 1) and tuple(shapes[1]) == (3, 1)
assert ((p_batch > 0) & (p_batch < 1)).all()
print("✅ Bài 4 đúng")

## Bài 5. Lớp `logistic_regression` tự viết

Viết lớp `logistic_regression(nn.Module)`: `__init__(self, n_inputs)` tạo `self.linear = nn.Linear(n_inputs, 1)`; `forward` trả `torch.sigmoid(self.linear(x))`.
- `m1 = logistic_regression(1)`, `keys_m1` là key của `state_dict`, `out1 = m1(X1)` với `X1 = [[-100.], [0.], [100.]]`.
- `m2 = logistic_regression(2)`, `out2 = m2(X2)` với `X2 = [[100., -100.], [0., 0.], [-100., 100.]]`.

In [ ]:
class logistic_regression(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.linear = nn.Linear(n_inputs, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

X1 = torch.tensor([[-100.0], [0.0], [100.0]])
X2 = torch.tensor([[100.0, -100.0], [0.0, 0.0], [-100.0, 100.0]])
m1 = logistic_regression(1)
keys_m1 = list(m1.state_dict().keys())
out1 = m1(X1)
m2 = logistic_regression(2)
out2 = m2(X2)
print(keys_m1, out1.shape, out2.shape)

In [ ]:
assert keys_m1 == ["linear.weight", "linear.bias"]
assert tuple(out1.shape) == (3, 1) and tuple(out2.shape) == (3, 1)
assert ((out1 >= 0) & (out1 <= 1)).all() and ((out2 >= 0) & (out2 <= 1)).all()
print("✅ Bài 5 đúng")

## Bài 6. Luyện tập: ghép hai khối bằng `nn.Sequential`

Tạo `my_model` = `nn.Sequential(...)` gồm: một lớp **tuyến tính** nhận 1 đặc trưng và cho ra 2 giá trị, **rồi** một `logistic_regression(2)`.
Dự đoán cho `X = [[-10.0]]` và lưu vào `my_pred`. Lưu `n_params` = tổng số phần tử tham số của `my_model`.

In [ ]:
my_model = nn.Sequential(nn.Linear(1, 2), logistic_regression(2))
my_pred = my_model(torch.tensor([[-10.0]]))
n_params = sum(p.numel() for p in my_model.parameters())
print(my_pred, n_params)

In [ ]:
assert tuple(my_pred.shape) == (1, 1) and 0 < my_pred.item() < 1
assert n_params == (1 * 2 + 2) + (2 * 1 + 1) == 7
print("✅ Bài 6 đúng: đây đã là hình dạng của một mạng nơ-ron nhỏ")

## Bài 7. Decision boundary của mô hình một đặc trưng

Tạo `model_b = logistic_regression(1)` và **đặt tay** `weight = 2`, `bias = -1` (bằng `with torch.no_grad(): ...copy_(...)`).
- `boundary = -b / w`: điểm ranh giới (số Python), tính từ tham số bạn đặt;
- `p_left`, `p_right`: xác suất tại `x = boundary - 0.1` và `x = boundary + 0.1` (số Python);
- `p_at`: xác suất đúng tại `x = boundary`.

In [ ]:
model_b = logistic_regression(1)
with torch.no_grad():
    model_b.linear.weight.copy_(torch.tensor([[2.0]]))
    model_b.linear.bias.copy_(torch.tensor([-1.0]))
    boundary = -model_b.linear.bias.item() / model_b.linear.weight.item()
    p_left = model_b(torch.tensor([[boundary - 0.1]])).item()
    p_right = model_b(torch.tensor([[boundary + 0.1]])).item()
    p_at = model_b(torch.tensor([[boundary]])).item()
print(boundary, p_left, p_right, p_at)

In [ ]:
assert abs(boundary - 0.5) < 1e-6
assert p_left < 0.5 < p_right, "hai phía ranh giới nhận hai lớp khác nhau"
assert abs(p_at - 0.5) < 1e-6, "đúng tại ranh giới, xác suất là 0.5"
print("✅ Bài 7 đúng")

## Bài 8. Độ tự tin tăng khi rời xa ranh giới

Với mô hình `weight = 2`, `bias = 0` (ranh giới ở `x = 0`): tính xác suất `p` tại `x = [0.01, 0.5, 3.0]` (tensor `(3, 1)`), rồi `confidence = |p − 0.5|`.
Lưu `ordered = True` nếu `confidence` **tăng dần** theo `x`.
Câu hỏi: ý nghĩa của `confidence` là gì?

In [ ]:
model_c = logistic_regression(1)
with torch.no_grad():
    model_c.linear.weight.copy_(torch.tensor([[2.0]]))
    model_c.linear.bias.copy_(torch.tensor([0.0]))
    p = model_c(torch.tensor([[0.01], [0.5], [3.0]]))
confidence = (p - 0.5).abs()
ordered = bool(torch.all(confidence[1:] > confidence[:-1]))
print(p.flatten(), confidence.flatten(), ordered)

In [ ]:
assert abs(p[0].item() - 0.505) < 0.001
assert p[2].item() > 0.99
assert ordered is True
print("✅ Bài 8 đúng: càng xa ranh giới, mô hình càng tự tin")

## Hết